# Deteccion de Outliers y Anomalias

_Detectar valores extremos y anomalias como parte de la limpieza de datos: metodos estadisticos (z-score, IQR, MAD) y basados en ML (Isolation Forest, DBSCAN)._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Deteccion de Outliers](assets/header.png)


## Introduccion

Un **outlier** es una observacion que se aparta notablemente del grueso de los
datos. Pueden ser **errores** (un sensor que falla, un dedo gordo al teclear) o
**senales reales pero raras** (una mansion en un barrio modesto, una venta atipica).
En feature engineering importan por dos razones:

- **Distorsionan estadisticos y modelos.** La media, la desviacion estandar, la
  regresion lineal o el escalado min-max son *muy* sensibles a valores extremos: un
  solo outlier puede arrastrar la media o estirar el rango.
- **Limpiar datos es parte del pipeline.** Antes de transformar y escalar conviene
  decidir que hacer con los extremos: dejarlos, recortarlos o transformarlos.

**El ejemplo canonico de Ames.** El dataset Ames Housing tiene unos outliers tan
famosos que el propio autor del dataset recomienda quitarlos: en el scatter de
`GrLivArea` (superficie habitable) vs `SalePrice`, hay **cuatro casas con
`GrLivArea > 4000`**. Dos de ellas son **ventas parciales** (`SaleCondition =
Partial`) que se vendieron *baratas* pese a ser enormes — rompen la relacion
"mas grande = mas cara". Las usamos como hilo conductor del notebook.

Veremos dos familias de metodos para *detectarlos* y luego *que hacer* con ellos:

1. **Metodos estadisticos** (univariados): z-score, regla IQR / vallas de Tukey,
   z-score modificado con MAD.
2. **Metodos basados en ML** (multivariados): Isolation Forest y DBSCAN
   (deteccion por densidad).
3. **Que hacer con los outliers**: eliminar, recortar / winsorizar, o transformar.

> **Outlier vs anomalia.** A escala univariada solemos hablar de *outliers*
> (valores extremos en una columna); cuando la rareza es *multivariada* (una
> combinacion inusual de varias variables, aunque cada una por separado sea normal)
> hablamos de *anomalias*, y ahi brillan los metodos basados en ML.


## Setup y datos

Cargamos Ames Housing (con fallback sintetico). Las columnas `GrLivArea`,
`LotArea` y `SalePrice` tienen colas y valores extremos reales, ideales para
ilustrar la deteccion de outliers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)


def load_housing():
    """Localiza housing_train.csv buscando rutas candidatas (offline-friendly).

    Busca, en orden: la variable de entorno HOUSING_CSV y varias rutas relativas
    al notebook. Si no encuentra el CSV, construye un pequeno DataFrame sintetico
    con las columnas clave para que el notebook siga corriendo. Imprime que fuente
    se uso.
    """
    import os
    import numpy as np
    import pandas as pd

    candidates = []
    if os.environ.get("HOUSING_CSV"):
        candidates.append(os.environ["HOUSING_CSV"])
    candidates += [
        os.path.join("..", "..", "data", "housing_train.csv"),
        os.path.join("..", "..", "..", "data", "housing_train.csv"),
        os.path.join("data", "housing_train.csv"),
    ]
    for path in candidates:
        if path and os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Dataset Ames Housing cargado desde: {path}  shape={df.shape}")
            return df

    # Fallback sintetico con las columnas clave (suficiente para que corra el codigo).
    print("CSV no encontrado: usando datos sinteticos con las columnas clave.")
    rng = np.random.default_rng(0)
    n = 400
    neighborhoods = [
        "NAmes", "CollgCr", "OldTown", "Edwards", "Somerst", "Gilbert",
        "NridgHt", "Sawyer", "NWAmes", "BrkSide", "Crawfor", "Mitchel",
    ]
    quality_levels = ["Po", "Fa", "TA", "Gd", "Ex"]

    def choice_with_na(levels, prob_na):
        """rng.choice sobre categorias string, inyectando NaN sin mezclar dtypes."""
        vals = np.array(rng.choice(levels, n), dtype=object)
        vals[rng.random(n) < prob_na] = np.nan
        return vals

    def present_or_na(value, prob_na):
        """Columna string donde NaN = 'feature ausente' (sin mezclar dtypes)."""
        vals = np.array([value] * n, dtype=object)
        vals[rng.random(n) < prob_na] = np.nan
        return vals

    overall_qual = rng.integers(3, 10, n)
    gr_liv_area = (overall_qual * 180 + rng.normal(400, 250, n)).clip(400, 5500)
    df = pd.DataFrame({
        "Id": np.arange(1, n + 1),
        "OverallQual": overall_qual,
        "GrLivArea": gr_liv_area.round().astype(int),
        "TotalBsmtSF": (gr_liv_area * 0.6 + rng.normal(0, 150, n)).clip(0, 3000).round().astype(int),
        "1stFlrSF": (gr_liv_area * 0.65 + rng.normal(0, 120, n)).clip(300, 3000).round().astype(int),
        "GarageCars": rng.integers(0, 4, n),
        "GarageArea": rng.integers(0, 900, n),
        "YearBuilt": rng.integers(1900, 2010, n),
        "LotArea": rng.gamma(2.0, 5000, n).clip(1300, 60000).round().astype(int),
        "LotFrontage": np.where(rng.random(n) < 0.18, np.nan, rng.integers(30, 120, n)).astype(float),
        "FullBath": rng.integers(0, 4, n),
        "Neighborhood": rng.choice(neighborhoods, n),
        "MSZoning": rng.choice(["RL", "RM", "FV", "RH", "C (all)"], n, p=[0.7, 0.15, 0.08, 0.04, 0.03]),
        "BldgType": rng.choice(["1Fam", "TwnhsE", "Duplex", "Twnhs", "2fmCon"], n),
        "Foundation": rng.choice(["PConc", "CBlock", "BrkTil", "Slab", "Stone", "Wood"], n),
        "CentralAir": rng.choice(["Y", "N"], n, p=[0.93, 0.07]),
        "ExterQual": rng.choice(quality_levels, n, p=[0.0, 0.0, 0.6, 0.35, 0.05]),
        "BsmtQual": choice_with_na(quality_levels, 0.03),
        "KitchenQual": rng.choice(quality_levels, n, p=[0.0, 0.05, 0.5, 0.4, 0.05]),
        "HeatingQC": rng.choice(quality_levels, n),
        "FireplaceQu": choice_with_na(quality_levels, 0.47),
        "Electrical": rng.choice(["SBrkr", "FuseA", "FuseF"], n),
        "Alley": present_or_na("Grvl", 0.93),
        "PoolQC": present_or_na("Gd", 0.995),
        "GarageType": present_or_na("Attchd", 0.06),
    })
    sale = (overall_qual * 22000 + df["GrLivArea"] * 55
            + rng.normal(0, 25000, n)).clip(40000, 600000)
    df["SalePrice"] = sale.round().astype(int)
    return df


df = load_housing()
gr_liv = df["GrLivArea"].to_numpy(dtype=float)
lot_area = df["LotArea"].to_numpy(dtype=float)
price = df["SalePrice"].to_numpy(dtype=float)
print("GrLivArea:", gr_liv.shape, "| LotArea:", lot_area.shape, "| SalePrice:", price.shape)
df[["GrLivArea", "LotArea", "SalePrice"]].describe().round(1)

## 0. Los outliers famosos de Ames: GrLivArea vs SalePrice

Antes de cualquier metrica, **mira los datos**. El scatter de superficie habitable
contra precio muestra una relacion creciente clara... salvo por cuatro casas
gigantes (`GrLivArea > 4000`) abajo a la derecha: enormes pero *baratas*. Dos son
ventas parciales (`Partial`). Son el ejemplo de libro de outliers que conviene
quitar antes de entrenar un modelo de precio.

In [ ]:
big = df["GrLivArea"] > 4000
print(f"Casas con GrLivArea > 4000: {int(big.sum())}")
cols = ["GrLivArea", "SalePrice"]
if "SaleCondition" in df.columns:
    cols.append("SaleCondition")
print(df.loc[big, cols].to_string())

plt.figure(figsize=(7, 5))
plt.scatter(df.loc[~big, "GrLivArea"], df.loc[~big, "SalePrice"],
            s=12, c="#4c78a8", alpha=0.6, label="normal")
plt.scatter(df.loc[big, "GrLivArea"], df.loc[big, "SalePrice"],
            s=70, c="#e45756", marker="x", label="GrLivArea > 4000")
plt.axvline(4000, ls="--", c="gray", lw=1)
plt.xlabel("GrLivArea (sqft)"); plt.ylabel("SalePrice")
plt.title("Outliers famosos de Ames: casas enormes vendidas baratas")
plt.legend(); plt.tight_layout(); plt.show()

## 1. Metodos estadisticos (univariados)

### 1a. Regla del z-score

**Intuicion primero.** Estandarizamos cada valor (cuantas desviaciones estandar se
aleja de la media) y marcamos como outlier todo lo que cae mas alla de un umbral,
tipicamente $|z| > 3$.

$$z_i = \frac{x_i - \mu}{\sigma}, \qquad \text{outlier si } |z_i| > 3.$$

**Cuidado.** La media $\mu$ y la desviacion $\sigma$ son *ellas mismas* sensibles a
los outliers: unos pocos extremos inflan $\sigma$ y "esconden" a los demas. Sirve
bien cuando los datos son aproximadamente **normales** y los extremos son pocos.
Lo aplicamos a `GrLivArea`.

In [ ]:
x = gr_liv
mu, sigma = x.mean(), x.std()
z = (x - mu) / sigma
out_z = np.abs(z) > 3
print(f"GrLivArea: media={mu:.1f}  std={sigma:.1f}")
print(f"z-score (|z|>3): {out_z.sum()} outliers de {len(x)}")
print("valores marcados:", np.sort(x[out_z])[-8:].round(0))

### 1b. Regla IQR / vallas de Tukey

**Intuicion primero.** En vez de media y desviacion (sensibles), usamos
**cuartiles**, que son robustos. El **rango intercuartil** $\text{IQR} = Q_3 - Q_1$
mide la dispersion del 50% central. Marcamos como outlier todo lo que cae fuera de
las **vallas de Tukey**:

$$[\,Q_1 - 1.5\,\text{IQR}, \;\; Q_3 + 1.5\,\text{IQR}\,].$$

(Con $3\,\text{IQR}$ en lugar de $1.5$ se marcan solo los outliers "extremos".) Es
el criterio que dibuja el clasico **boxplot**: los bigotes llegan hasta las vallas
y los puntos mas alla se grafican como outliers. Lo mostramos sobre `LotArea`, una
columna con colas muy pesadas (lotes enormes).

In [ ]:
def iqr_fences(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

low, high = iqr_fences(lot_area)
out_iqr = (lot_area < low) | (lot_area > high)
print(f"LotArea: Q1={np.percentile(lot_area,25):.0f}  Q3={np.percentile(lot_area,75):.0f}  "
      f"IQR={np.percentile(lot_area,75)-np.percentile(lot_area,25):.0f}")
print(f"Vallas de Tukey: [{low:.0f}, {high:.0f}]")
print(f"IQR (1.5x): {out_iqr.sum()} outliers de {len(lot_area)}")

In [ ]:
# Boxplot: los puntos mas alla de los bigotes son los outliers segun la regla IQR.
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot(lot_area, vert=True, widths=0.5)
ax[0].set_title("Boxplot de 'LotArea'\n(puntos = outliers por IQR)"); ax[0].set_ylabel("LotArea")

# Scatter resaltando los outliers IQR sobre el indice.
ax[1].scatter(np.arange(len(lot_area))[~out_iqr], lot_area[~out_iqr], s=10,
              c="#4c78a8", label="normal")
ax[1].scatter(np.arange(len(lot_area))[out_iqr], lot_area[out_iqr], s=28,
              c="#e45756", label="outlier IQR")
ax[1].axhline(high, ls="--", c="gray", lw=1, label="valla superior")
ax[1].set_title("'LotArea' con outliers resaltados"); ax[1].set_xlabel("indice")
ax[1].set_ylabel("LotArea"); ax[1].legend()
plt.tight_layout(); plt.show()

### 1c. z-score modificado (con MAD)

**Intuicion primero.** Arreglamos la fragilidad del z-score clasico cambiando media
y desviacion por sus versiones **robustas**: la **mediana** y la **MAD** (mediana de
las desviaciones absolutas respecto a la mediana).

$$\text{MAD} = \text{mediana}\big(|x_i - \tilde{x}|\big), \qquad
M_i = \frac{0.6745\,(x_i - \tilde{x})}{\text{MAD}}.$$

El factor $0.6745$ hace que $M$ sea comparable a un z-score bajo normalidad. Una
regla comun: outlier si $|M_i| > 3.5$. Como la mediana y la MAD apenas se mueven
con los extremos, este criterio **no se deja enganar** por los propios outliers.
Lo aplicamos a `SalePrice`.

In [ ]:
def modified_zscore(x):
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad == 0:
        return np.zeros_like(x, dtype=float)
    return 0.6745 * (x - med) / mad

m = modified_zscore(price)
out_mad = np.abs(m) > 3.5
print(f"SalePrice: mediana={np.median(price):.0f}  "
      f"MAD={np.median(np.abs(price-np.median(price))):.0f}")
print(f"z-score modificado (|M|>3.5): {out_mad.sum()} outliers")

# Comparacion de los tres criterios univariados sobre SalePrice.
z_price = np.abs((price - price.mean()) / price.std()) > 3
low_p, high_p = iqr_fences(price)
iqr_price = (price < low_p) | (price > high_p)
print("\nComparacion de conteos sobre 'SalePrice':")
print(f"  z-score clasico  (|z|>3)  : {z_price.sum()}")
print(f"  IQR / Tukey      (1.5xIQR): {iqr_price.sum()}")
print(f"  z-score modif.   (|M|>3.5): {out_mad.sum()}")

## 2. Metodos basados en ML (multivariados)

Los criterios anteriores miran **una columna a la vez**. Pero una observacion puede
ser normal en cada variable por separado y aun asi ser una **combinacion** rara —
exactamente el caso de las casas grandes-y-baratas de Ames. Para eso usamos
metodos multivariados sobre el plano 2-D **`[GrLivArea, SalePrice]`**
(estandarizado). Como "verdad" aproximada marcamos las casas con `GrLivArea > 4000`,
para poder comparar los metodos por ROC-AUC.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Espacio 2-D: superficie habitable vs precio. Estandarizamos (los metodos miden
# distancias / cortes y las dos columnas viven en escalas muy distintas).
X2 = df[["GrLivArea", "SalePrice"]].to_numpy(dtype=float)
X = StandardScaler().fit_transform(X2)

# "Verdad" aproximada: las 4 casas gigantes (GrLivArea>4000) son los outliers conocidos.
y_true = (df["GrLivArea"] > 4000).astype(int).to_numpy()
contamination = max(y_true.mean(), 1e-3)
print(f"X: {X.shape} | outliers conocidos (GrLivArea>4000): {int(y_true.sum())} | "
      f"contaminacion: {contamination:.4f}")

plt.figure(figsize=(6.5, 5))
plt.scatter(X[y_true == 0, 0], X[y_true == 0, 1], s=12, c="tab:blue", label="normal")
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], s=60, c="tab:red", marker="x",
            label="GrLivArea > 4000")
plt.xlabel("GrLivArea (estandarizado)"); plt.ylabel("SalePrice (estandarizado)")
plt.legend(); plt.title("Plano [GrLivArea, SalePrice] estandarizado")
plt.tight_layout(); plt.show()

### 2a. Isolation Forest

**Intuicion primero.** Las anomalias son *pocas y distintas*, asi que son **faciles
de aislar**. Un *iTree* particiona los datos eligiendo una **variable aleatoria** y
un **corte aleatorio**. Las anomalias, en regiones esparsas, quedan separadas tras
**muy pocos cortes** (camino corto desde la raiz); los inliers, en regiones densas,
necesitan **muchos** cortes (camino largo).

El score usa la **longitud de camino esperada** $E(h(x))$ sobre el ensemble,
normalizada por la longitud media de una busqueda fallida en un BST de $n$ puntos,

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \qquad H(i)\approx \ln(i) + 0.5772,$$

dando

$$s(x, n) = 2^{-\,E(h(x)) / c(n)}.$$

$s \to 1$: caminos cortos → **anomalia**; $s \to 0.5$: cerca de la media → normal.
Es rapido ($O(n\log n)$), maneja alta dimension y **no computa distancias**.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

iforest = IsolationForest(n_estimators=200, contamination=contamination,
                          random_state=42)
iforest.fit(X)
# decision_function: mayor = mas normal. Lo negamos para que mayor = mas anomalo.
if_scores = -iforest.decision_function(X)
if_auc = roc_auc_score(y_true, if_scores)
print(f"Isolation Forest ROC-AUC (vs casas >4000 sqft): {if_auc:.4f}")
print("Top-6 casas mas anomalas segun IF (indice, GrLivArea, SalePrice):")
top = np.argsort(if_scores)[::-1][:6]
print(df.iloc[top][["GrLivArea", "SalePrice"]].to_string())

### 2b. DBSCAN (deteccion por densidad)

**Intuicion primero.** DBSCAN agrupa los puntos en regiones **densas** y deja
fuera lo que no encaja: los puntos que no pertenecen a ningun cluster se etiquetan
como **ruido** (`label = -1`), y ese ruido son justamente nuestros **outliers**. No
hay que decirle cuantos clusters hay — la **densidad** lo decide.

Dos hiperparametros mandan:

- **`eps`**: radio del vecindario. Un punto es **nucleo (core)** si tiene al menos
  `min_samples` vecinos dentro de `eps`.
- **`min_samples`**: cuantos vecinos definen una region densa.

Un punto que no es nucleo ni **alcanzable** desde un nucleo queda como
**ruido / outlier**. DBSCAN encuentra anomalias **globales y de forma arbitraria**
(no asume blobs gaussianos), pero es sensible a la **escala** (ya estandarizamos) y
sobre todo a `eps`.

In [ ]:
from sklearn.cluster import DBSCAN

# Ya trabajamos sobre X estandarizado.
dbscan = DBSCAN(eps=0.5, min_samples=10).fit(X)
labels = dbscan.labels_                 # -1 = ruido (outlier)
db_outlier = (labels == -1)

# DBSCAN da una etiqueta DURA (no un score): usamos el flag binario como "score".
db_auc = roc_auc_score(y_true, db_outlier.astype(float))
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"DBSCAN — clusters densos: {n_clusters} | ruido (outliers): {db_outlier.sum()}  "
      f"| ROC-AUC: {db_auc:.4f}")

plt.figure(figsize=(6.5, 5.5))
plt.scatter(X[~db_outlier, 0], X[~db_outlier, 1], s=14, c=labels[~db_outlier],
            cmap="tab10", label="clusters densos")
plt.scatter(X[db_outlier, 0], X[db_outlier, 1], s=45, c="red", marker="x",
            label="ruido = outlier")
plt.legend(loc="upper left")
plt.xlabel("GrLivArea (estandarizado)"); plt.ylabel("SalePrice (estandarizado)")
plt.title("DBSCAN: los puntos de ruido (-1) son outliers")
plt.tight_layout(); plt.show()

### Vista comparada: Isolation Forest vs DBSCAN

A la izquierda, los contornos del **score** de Isolation Forest con su **frontera**
(linea discontinua). A la derecha, los **clusters densos** de DBSCAN con el ruido
(`-1`) resaltado. IF entrega un *score continuo* por punto; DBSCAN, una *etiqueta
dura* (cluster vs ruido).

In [ ]:
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400), np.linspace(y_min, y_max, 400))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
# Izquierda: Isolation Forest (score + frontera)
Z = iforest.decision_function(grid).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, levels=25, cmap="RdBu")
axes[0].contour(xx, yy, Z, levels=[0], colors="k", linestyles="dashed", linewidths=1.5)
axes[0].scatter(X[y_true == 0, 0], X[y_true == 0, 1], s=10, c="white",
                edgecolor="k", lw=0.3, label="normal")
axes[0].scatter(X[y_true == 1, 0], X[y_true == 1, 1], s=40, c="yellow",
                edgecolor="k", marker="x", label="GrLivArea>4000")
axes[0].set_title("Isolation Forest — score y frontera")
axes[0].set_xlabel("GrLivArea (z)"); axes[0].set_ylabel("SalePrice (z)")
axes[0].legend(loc="upper left", fontsize=8)
# Derecha: DBSCAN (clusters densos vs ruido)
axes[1].scatter(X[~db_outlier, 0], X[~db_outlier, 1], s=12,
                c=labels[~db_outlier], cmap="tab10")
axes[1].scatter(X[db_outlier, 0], X[db_outlier, 1], s=45, c="red", marker="x",
                label="ruido = outlier")
axes[1].set_title("DBSCAN — clusters densos vs ruido (-1)")
axes[1].set_xlabel("GrLivArea (z)"); axes[1].set_ylabel("SalePrice (z)")
axes[1].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

### Comparativa y cuando usar cada metodo ML

| Aspecto | **Isolation Forest** | **DBSCAN** |
|---|---|---|
| Principio | aislamiento aleatorio, longitud de camino | densidad: nucleos, alcanzabilidad y ruido |
| Salida | **score continuo** de anomalia | **etiqueta dura** (cluster vs ruido `-1`) |
| Tipo de outlier | global, multivariado | global, **forma arbitraria** (no asume blobs) |
| Tamano de datos | **grande**, $O(n\log n)$ | pequeno/medio (coste de vecindarios) |
| Knobs clave | `n_estimators`, `contamination` | `eps`, `min_samples` (y **escala**) |

**Reglas de oro**

- **Datos grandes / alta dimension, baseline rapido y con score →** Isolation Forest.
- **Clusteres de densidad o forma arbitraria, sin fijar cuantos hay →** DBSCAN
  (estandariza y ajusta `eps`).

In [ ]:
comparison = pd.DataFrame({
    "IsolationForest": {"roc_auc": if_auc},
    "DBSCAN": {"roc_auc": db_auc},
}).T.sort_values("roc_auc", ascending=False)
print("ROC-AUC sobre los outliers conocidos (casas GrLivArea>4000):")
print(comparison.round(4))

## 3. Que hacer con los outliers

Detectarlos es solo la mitad: hay que **decidir que hacer**. Tres estrategias
principales, con criterios para elegir:

- **Eliminar (drop).** Quita las filas marcadas. En Ames, el caso de libro es
  **eliminar las 4 casas con `GrLivArea > 4000`**: el propio autor del dataset lo
  recomienda porque son ventas atipicas que distorsionan la relacion area-precio.
  Solo elimina si estas convencido de que son **errores o casos atipicos** y son
  **pocos**; tirar muchas filas sesga el dataset.
- **Recortar / winsorizar (cap).** En vez de borrar, **limitas** los valores
  extremos a un percentil (p. ej. todo por encima del P99 pasa a valer el P99).
  Conserva la fila y su informacion, solo domestica la magnitud. Buen default
  cuando los extremos son reales pero distorsionan.
- **Transformar.** Aplicar `log` / Box-Cox (Notebook 1) **comprime las colas**, asi
  que muchos "outliers" dejan de serlo sin borrar nada. Ideal para variables
  positivas y sesgadas como `SalePrice` o `LotArea`.

La eleccion depende de *por que* existe el outlier (error vs senal), *cuantos* hay
y del *modelo* (los arboles toleran extremos; los lineales/distancia, no).

In [ ]:
# DROP: el tratamiento de libro en Ames -> quitar las 4 casas GrLivArea>4000.
n_before = len(df)
df_clean = df[df["GrLivArea"] <= 4000].copy()
print(f"DROP casas GrLivArea>4000: {n_before} -> {len(df_clean)} filas "
      f"({n_before - len(df_clean)} eliminadas)")

# WINSORIZAR: capar 'LotArea' a los percentiles 1 y 99.
p1, p99 = np.percentile(lot_area, [1, 99])
lot_wins = np.clip(lot_area, p1, p99)
print(f"\nWinsorizado LotArea a [P1={p1:.0f}, P99={p99:.0f}]")
print(f"max antes={lot_area.max():.0f}  ->  max despues={lot_wins.max():.0f}")

# TRANSFORMAR: log1p comprime la cola (muchos outliers dejan de serlo).
lot_log = np.log1p(lot_area)
low_l, high_l = iqr_fences(lot_log)
out_log = (lot_log < low_l) | (lot_log > high_l)
print(f"\nOutliers IQR en LotArea antes de log : {out_iqr.sum()}")
print(f"Outliers IQR en LotArea tras log1p   : {out_log.sum()}  (la transformacion los reduce)")

In [ ]:
# Comparacion visual de las tres estrategias sobre 'LotArea'.
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(lot_area, bins=40, color="#e45756"); ax[0].set_title("Original (con outliers)")
ax[1].hist(lot_wins, bins=40, color="#f0a500"); ax[1].set_title("Winsorizado (cap P1-P99)")
ax[2].hist(lot_log, bins=40, color="#4c78a8"); ax[2].set_title("Transformado (log1p)")
for a in ax: a.set_xlabel("valor"); a.set_ylabel("conteo")
plt.tight_layout(); plt.show()

## Resumen — tabla de referencia rapida

| Metodo | Tipo | Idea | Robusto a outliers | Usar cuando |
|---|---|---|---|---|
| **z-score** | estadistico | $|z|>3$ con media/desv. | no | datos ~normales, pocos extremos |
| **IQR / Tukey** | estadistico | fuera de $Q_1-1.5\,IQR$, $Q_3+1.5\,IQR$ | si | default univariado, base del boxplot |
| **z-score modif. (MAD)** | estadistico | mediana + MAD, $|M|>3.5$ | si | colas pesadas, criterio robusto |
| **Isolation Forest** | ML | aislamiento por cortes aleatorios | n/a | multivariado, datos grandes, score continuo |
| **DBSCAN** | ML | densidad: nucleos + ruido (`-1`) | n/a | clusters de forma arbitraria, anomalias globales |

**Que hacer despues:** *eliminar* si son errores o casos atipicos y son pocos (las 4
casas `GrLivArea>4000` de Ames son el ejemplo de libro); *winsorizar* si son reales
pero distorsionan; *transformar* (log / Box-Cox) para comprimir colas sin borrar
informacion. La eleccion depende del origen del outlier, su cantidad y el modelo.

> En el **Notebook 3** pasamos de la limpieza al **feature store** (Feast): como
> servir estas features de forma consistente entre entrenamiento y produccion.